In [2]:
#Import the needed libraries.
import pandas as pd
import streamlit as st
from rapidfuzz import process, fuzz

In [103]:
#Load the data
df = pd.read_csv('amazon-python.csv')

In [104]:
#Check what columns i have
df.columns

Index(['product_id', 'product_name', 'category', 'discounted_price',
       'actual_price', 'discount_percentage', 'rating', 'rating_count',
       'about_product', 'user_id', 'user_name', 'review_id', 'review_title',
       'review_content', 'img_link', 'product_link'],
      dtype='str')

In [105]:
#Look at the first 3 rows to verify the data isn't corrupted
df.head(3)

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...


In [106]:
#Replace '_' with a space.
df.columns = df.columns.str.replace('_',' ')

In [107]:
#Convert Columns Titles To Title Case.
df.columns = df.columns.str.title()

In [108]:
#Check the fix
df.columns

Index(['Product Id', 'Product Name', 'Category', 'Discounted Price',
       'Actual Price', 'Discount Percentage', 'Rating', 'Rating Count',
       'About Product', 'User Id', 'User Name', 'Review Id', 'Review Title',
       'Review Content', 'Img Link', 'Product Link'],
      dtype='str')

In [109]:
#List all columns
print(df.columns.tolist())

['Product Id', 'Product Name', 'Category', 'Discounted Price', 'Actual Price', 'Discount Percentage', 'Rating', 'Rating Count', 'About Product', 'User Id', 'User Name', 'Review Id', 'Review Title', 'Review Content', 'Img Link', 'Product Link']


In [110]:
#There are some columns I do not need in my analysis, so I will remove them.

In [111]:
# Drop unneeded columns
columns_to_drop = ['Product Link', 'Img Link', 'Review Content', 'Review Title', 'About Product']
df = df.drop(columns=columns_to_drop)

In [112]:
#Verify the changes
df.columns

Index(['Product Id', 'Product Name', 'Category', 'Discounted Price',
       'Actual Price', 'Discount Percentage', 'Rating', 'Rating Count',
       'User Id', 'User Name', 'Review Id'],
      dtype='str')

In [113]:
# Inspect unique values in 'User Name' to spot placeholders or missing data
print(df['User Name'].value_counts(dropna=False))

User Name
$@|\|TO$|-|,Sethu madhav,Akash Thakur,Burger Planet,Justice ⚖️,indrajyoti d.,Aditya Kumar,E.C.GEORGE                                         10
Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jaspreet singh,Khaja moin,Anand,S.ARUMUGAM                                                          8
Satheesh Kadiam,Pritom Chakraborty,Vishwa,Simranpreet Singh,Saptarshi,Amazon Customer,D.RAGHUL,Dharmendra kumar                               8
ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Placeholder,BharanI,sonia,Niam                                                                      7
Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh chouhan,Ravi Siddan,Himanshu Goel,Udaykumar                                                     7
Prashant,Sumesh Sundararajan,Vijay Baitha,S.k nahak,Vikram Kumar,Manish,Jm,amit nayak                                                         6
siddharth patnaik,Dr Sunilkumar H,Krishna,K. S. Rao,vinayp,indhu,Jogi,DRISHTI VASHISTH GUPTA                                  

In [114]:
#cells aren't single names — 
#they're comma-jammed blobs like:"Amazon Customer,Sethu madhav,Akash Thakur"

In [115]:
#I need to show all the rows instead of a sample of them.
pd.set_option('display.max_rows',None)

In [117]:
# Verify the changes
#I want to see all the names separately, each name on its own line
all_names = df['User Name'].dropna().astype(str).str.split(',').explode().str.strip()
names = all_names.value_counts(dropna=False).head(300)
for name in names.index:
    print(name)

Amazon Customer
Placeholder
Kindle Customer
Rajesh
Deepak
Rohit
Arun
Prashant
Amit
Anand
Manoj Kumar
Vivek
Aditya Kumar
Karthik
Krishna
Sethu madhav
E.C.GEORGE
Ankit
Abhishek
Manav
ArdKn
Shankar
Rakesh
Mahesh
$@|\|TO$|-|
Akash Thakur
Burger Planet
Justice ⚖️
indrajyoti d.
Ajay
Pankaj
Manish
S.ARUMUGAM
Akshay
Santhosh
Jay
Customer
User
Pradeep
Vinay
Rahul
Ravi
Sameer
Adarsh gupta
Sundeep
S.Sayeed Ahmed
jaspreet singh
Khaja moin
Himanshu
JD
Jayesh
Aman
Priya
Ashutosh
Mohan
Naveen
Siddharth
Anonymous
Vijay
Nitin
Satheesh Kadiam
Pritom Chakraborty
Vishwa
Simranpreet Singh
Saptarshi
D.RAGHUL
Dharmendra kumar
Nirbhay kumar
Sagar Viswanathan
Asp
BharanI
sonia
Niam
Omkar dhale
HEMALATHA
Ajwadh a.
amar singh chouhan
Ravi Siddan
Himanshu Goel
Udaykumar
Harsha
Ayush
Neeraj Vishwakarma
Suriya
Ganesh
Amrut K.
chetan tandel
Pranav
AV
indhu
Gaurav
Sunil
Rajesh k.
Soopy
amazon customer
dinesh
Chitra
Ajaybabu.O.M
Manoj
SP
Sumit
Mukundha2good
Shiva
arun
Gowthami
Mwnzil brahma
Pratik
Unknown
Vishal
YOGES

In [118]:
# Replace missing or generic user names with 'N/A' while keeping valid names unchanged.
df['User Name'] = df['User Name'].apply(
    lambda x: 'N/A' if pd.isna(x) or str(x).strip().lower() in[
        'placeholder',
        'amazon customer',
        'kindle customer',
        'unknown',
        ''
    ]else x
)

In [119]:
# Verify the changes
all_names = df['User Name'].dropna().astype(str).str.split(',').explode().str.strip()
names = all_names.value_counts(dropna=False).head(300)
for name in names.index:
    print(name)

Amazon Customer
Placeholder
Kindle Customer
Rajesh
Deepak
Rohit
Arun
Prashant
Amit
Anand
Manoj Kumar
Vivek
Aditya Kumar
Karthik
Krishna
Sethu madhav
E.C.GEORGE
Ankit
Abhishek
Manav
ArdKn
Shankar
Rakesh
Mahesh
$@|\|TO$|-|
Akash Thakur
Burger Planet
Justice ⚖️
indrajyoti d.
Ajay
Pankaj
Manish
S.ARUMUGAM
Akshay
Santhosh
Jay
Customer
User
Pradeep
Vinay
Rahul
Ravi
Sameer
Adarsh gupta
Sundeep
S.Sayeed Ahmed
jaspreet singh
Khaja moin
Himanshu
JD
Jayesh
Aman
Priya
Ashutosh
Mohan
Naveen
Siddharth
Anonymous
Vijay
Nitin
Satheesh Kadiam
Pritom Chakraborty
Vishwa
Simranpreet Singh
Saptarshi
D.RAGHUL
Dharmendra kumar
Nirbhay kumar
Sagar Viswanathan
Asp
BharanI
sonia
Niam
Omkar dhale
HEMALATHA
Ajwadh a.
amar singh chouhan
Ravi Siddan
Himanshu Goel
Udaykumar
Harsha
Ayush
Neeraj Vishwakarma
Suriya
Ganesh
Amrut K.
chetan tandel
Pranav
AV
indhu
Gaurav
Sunil
Rajesh k.
Soopy
amazon customer
dinesh
Chitra
Ajaybabu.O.M
Manoj
SP
Sumit
Mukundha2good
Shiva
arun
Gowthami
Mwnzil brahma
Pratik
Unknown
Vishal
YOGES

In [120]:
# df['User Name'] = df['User Name'].apply(
#     lambda x: 'N/A' if pd.isna(x) or str(x).strip().lower() in[
#         'placeholder',
#         'amazon customer',
#         'kindle customer',
#         'unknown',
#         ''
#     ]else x
# )
#The code does not work. There is something wrong with it!
#The Names did not change after I checked them.
#The likely cause: The User Name
#cells aren't single names — 
#they're comma-jammed blobs like:"Amazon Customer,Sethu madhav,Akash Thakur"

In [121]:
#Want to confirm this is the problem:
#If this returns 0 (or a suspiciously low number) 
#even though i know "Amazon Customer" appears in the data, 
#that confirms exact whole-cell matching is the blocker.
contains=df['User Name'].astype(str).str.strip().str.lower().str.contains('amazon customer').sum()
equal=df['User Name'].astype(str).str.strip().str.lower().eq('amazon customer').sum()
print(f'contains = {contains} and equal = {equal}')

contains = 529 and equal = 0


In [122]:
#As I expected, 
#the problem is that the User Name column cells aren't single names
#— they're comma-jammed blobs.

In [123]:
#I need to clean the data at the individual-name level,
#not the cell level, using the explode() approach.
generic_terms = ['placeholder', 'amazon customer', 'kindle customer', 'unknown', '']
def clean_names(cell):
    if pd.isna(cell):
        #Step 1: if the whole cell is empty/missing, just return 'N/A'
        return 'N/A'
    #Split the cell into individual names using the comma
    names_list = str(cell).split(',')
    #Go through each name one by one and check it
    cleaned_list=[]
    for name in names_list:
        #Remove extra spaces 
        name = name.strip()
        #lowercase version for comparison
        name_lower = name.casefold()

        if name_lower in generic_terms:
            # Replace generic name
            cleaned_list.append('N/A')
        else:
            # Keep the real name
            cleaned_list.append(name)
    #Join the cleaned names back together with commas
    return ','.join(cleaned_list)

df['User Name'] = df['User Name'].apply(clean_names)

In [151]:
# Verify the changes
# all_names = df['User Name'].dropna().astype(str).str.split(',').explode().str.strip()
# names = all_names.value_counts(dropna=False).head(300)
# for name in names.index:
#     print(name)
# contains=df['User Name'].astype(str).str.strip().str.lower().str.contains('amazon customer').sum()
# equal=df['User Name'].astype(str).str.strip().str.lower().eq('amazon customer').sum()
# print(f'contains = {contains} and equal = {equal}')
# contains=df['User Name'].astype(str).str.strip().str.lower().str.contains('amazon customer').sum()
# equal=df['User Name'].astype(str).str.strip().str.lower().eq('amazon customer').sum()
# print(f'contains = {contains} and equal = {equal}')
# df.loc[
#     df['User Name'].astype(str).str.strip().str.lower().str.contains('amazon customer'),
#     'User Name'
# ]
#pd.set_option('display.max_colwidth', None)
#pd.set_option('display.max_columns', None)
# df['User Name'] = df['User Name'].str.replace(
#     'Amazon Customercare',
#     'N/A',
#     case=False,
#     regex=False
# )
# df['User Name'] = df['User Name'].str.replace(
#     'BOSSAmazon Customer',
#     'N/A',
#     case=False,
#     regex=False
# )
# 
#After all the checks, the adjustments are done, so let's go to the next step.

In [160]:
df.to_csv('amazon-python.csv', index=False,encoding='utf-8-sig')

In [167]:
#Look at the Actual Price and Discounted Price values and their data types
print(df[['Actual Price','Discounted Price']].head())
print(df[['Actual Price', 'Discounted Price']].dtypes)

  Actual Price Discounted Price
0       ₹1,099             ₹399
1         ₹349             ₹199
2       ₹1,899             ₹199
3         ₹699             ₹329
4         ₹399             ₹154
Actual Price        str
Discounted Price    str
dtype: object


In [168]:
for col in ['Actual Price','Discounted Price']:
    # Remove the symbol and commas
    df[col] = df[col].str.replace('₹', '').str.replace(',', '')
    # Convert to numeric
    df[col] = pd.to_numeric(df[col])

In [169]:
# Verify the fix (they should now be float64)
print(df[['Actual Price', 'Discounted Price']].dtypes)

Actual Price        float64
Discounted Price    float64
dtype: object


In [173]:
# Verify the fix (Remove the symbol and commas)
print(df[['Actual Price','Discounted Price']].head())

   Actual Price  Discounted Price
0        1099.0             399.0
1         349.0             199.0
2        1899.0             199.0
3         699.0             329.0
4         399.0             154.0


In [176]:
#Look at the raw format of the Category column.
#I noticed that it contains a long chain of categories separated by a pipe symbol (|).
# Look at the raw format of the Category column
print(df['Category'].head())

0    Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables
1    Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables
2    Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables
3    Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables
4    Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables
Name: Category, dtype: str


In [177]:
#I will use the str.split() function to break that long text apart at every | symbol.
#Then, I will keep only the first two parts and rename them to Category and Sub Category.
# Split the text into multiple columns based on the '|' delimiter
#expand=True tells Pandas to put the split results into separate columns.
split_cat = df['Category'].str.split('|',expand=True)
# Keep the first part (index 0) as Category
df['Category'] = split_cat[0]
# Keep the second part (index 1) as Sub Category
df['Sub Category'] = split_cat[1]

In [179]:
# Verify the split worked cleanly
print(df[['Category', 'Sub Category']].head(3))

                Category             Sub Category
0  Computers&Accessories  Accessories&Peripherals
1  Computers&Accessories  Accessories&Peripherals
2  Computers&Accessories  Accessories&Peripherals


In [181]:
#All columns for now.
df.columns

Index(['Product Id', 'Product Name', 'Category', 'Discounted Price',
       'Actual Price', 'Discount Percentage', 'Rating', 'Rating Count',
       'User Id', 'User Name', 'Review Id', 'Sub Category'],
      dtype='str')

In [183]:
df.to_csv('amazon-python.csv',index=False,encoding='utf-8-sig')

In [195]:
# Look for messy spacing in product names
df['Product Name'].head(100).tolist()

['Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)',
 'Ambrane Unbreakable 60W / 3A Fast Charging 1.5m Braided Type C Cable for Smartphones, Tablets, Laptops & other Type C devices, PD Technology, 480Mbps Data Sync, Quick Charge 3.0 (RCT15A, Black)',
 'Sounce Fast Phone Charging Cable & Data Sync USB Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini & iOS Devices',
 'boAt Deuce USB 300 2 in 1 Type-C & Micro USB Stress Resistant, Tangle-Free, Sturdy Cable with 3A Fast Charging & 480mbps Data Transmission, 10000+ Bends Lifespan and Extended 1.5m Length(Martian Red)',
 'Portronics Konnect L 1.2M Fast Charging 3A 8 Pin USB Cable with Charge & Sync Function for iPhone, iPad (Grey)',
 'pTron Solero TB301 3A Type-C Data and Fast Charging Cable, Made in India, 480Mbps Data Sync, Strong and Durable 1.5-Meter Nylon Braided USB Cable for Type-C Devices for Char

In [198]:
#In this dataset, 
#the first word in the product name is most likely the brand name 
#— not for all of them, but for most of them.
# Look at what the raw first word looks like before cleaning
print(df['Product Name'].str.split(' ').str[0].value_counts().head(100))

Product Name
boAt            67
Samsung         36
AmazonBasics    33
Portronics      31
Ambrane         29
Redmi           26
Fire-Boltt      26
Bajaj           26
Amazon          25
Wayona          24
Noise           24
MI              21
HP              21
Duracell        20
Havells         19
Philips         18
TP-Link         17
SanDisk         17
Zebronics       16
Logitech        15
Crompton        15
ZEBRONICS       14
iQOO            14
AGARO           14
Prestige        14
pTron           13
OnePlus         13
KENT            13
Gizga           12
Classmate       12
Lifelong        12
Sounce          11
7SEVEN®         11
STRIFF          11
Boult           11
Pigeon          11
Lapster         10
Acer             9
Lenovo           8
JBL              8
Usha             8
Mi               7
FLiX             7
PTron            7
Belkin           7
Eureka           7
Wipro            7
iBELL            7
LG               6
Airtel           6
Storite          6
Nokia            6

In [199]:
# Clean product names 
#(Trim, Clean, and replace multiple spaces with one)
df['Product Name'] = df['Product Name'].str.strip().replace(r'\s+',' ',regex=True)

In [200]:
# Extract first word before the space into a new 'Brand' column
df['Brand'] = df['Product Name'].str.split(' ').str[0]

In [202]:
df.columns

Index(['Product Id', 'Product Name', 'Category', 'Discounted Price',
       'Actual Price', 'Discount Percentage', 'Rating', 'Rating Count',
       'User Id', 'User Name', 'Review Id', 'Sub Category', 'Brand'],
      dtype='str')

In [205]:
# Strip out trademark symbols and commas
df['Brand'] = df['Brand'].str.replace('™','',regex=False)
df['Brand'] = df['Brand'].str.replace('®','',regex=False)
df['Brand'] = df['Brand'].str.replace(',','',regex=False)

In [206]:
# Map specific messy names to their clean versions or "Unknown".
brand_map = {
    'iPhone': 'Apple', '3M': 'Unknown', '4': 'Unknown', '10k': 'Unknown',
    '!!HANEUL!!1000': 'Unknown', '10WeRun': 'WeRun', '!!1000': 'Unknown',
    'SoniVision': 'Soni', 'realme': 'Realme', 'USB': 'Oraimo', 'Amozo': 'Unknown',
    'Noise_Colorfit': 'Unknown', 'Brand': 'Unknown', 'TABLE': 'Unknown',
    'R': 'Unknown', 'Dr': 'Unknown', 'Tom': 'Unknown', 'Multifunctional': 'Unknown',
    'MUnknown.': 'Unknown', 'Unknownacold': 'Unknown', 'T': 'Unknown',
    'Empty': 'Unknown', 'Cafe': 'Unknown', 'SUJAUnknownA': 'Unknown',
    'CAUnknownDEX': 'CARDEX', 'Eco': 'Unknown', 'Monitor': 'Unknown',
    'Aqua': 'Unknown', 'Model-PUnknown': 'Unknown'
}
df['Brand'] = df['Brand'].replace(brand_map)

In [210]:
# Verify the clean product names and new brand column
df[['Product Name','Brand']].head(100)

,Product Name,Brand
0,"Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)",Wayona
1,"Ambrane Unbreakable 60W / 3A Fast Charging 1.5m Braided Type C Cable for Smartphones, Tablets, Laptops & other Type C devices, PD Technology, 480Mbps Data Sync, Quick Charge 3.0 (RCT15A, Black)",Ambrane
2,"Sounce Fast Phone Charging Cable & Data Sync USB Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini & iOS Devices",Sounce
3,"boAt Deuce USB 300 2 in 1 Type-C & Micro USB Stress Resistant, Tangle-Free, Sturdy Cable with 3A Fast Charging & 480mbps Data Transmission, 10000+ Bends Lifespan and Extended 1.5m Length(Martian Red)",boAt
4,"Portronics Konnect L 1.2M Fast Charging 3A 8 Pin USB Cable with Charge & Sync Function for iPhone, iPad (Grey)",Portronics
5,"pTron Solero TB301 3A Type-C Data and Fast Charging Cable, Made in India, 480Mbps Data Sync, Strong and Durable 1.5-Meter Nylon Braided USB Cable for Type-C Devices for Charging Adapter (Black)",pTron
6,"boAt Micro USB 55 Tangle-free, Sturdy Micro USB Cable with 3A Fast Charging & 480mbps Data Transmission (Black)",boAt
7,MI Usb Type-C Cable Smartphone (Black),MI
8,"TP-Link USB WiFi Adapter for PC(TL-WN725N), N150 Wireless Network Adapter for Desktop - Nano Size WiFi Dongle Compatible with Windows 11/10/7/8/8.1/XP/ Mac OS 10.9-10.15 Linux Kernel 2.6.18-4.4.3",TP-Link
9,"Ambrane Unbreakable 60W / 3A Fast Charging 1.5m Braided Micro USB Cable for Smartphones, Tablets, Laptops & Other Micro USB Devices, 480Mbps Data Sync, Quick Charge 3.0 (RCM15, Black)",Ambrane


In [211]:
df.to_csv('amazon-python.csv',index=False,encoding='utf-8-sig')

In [213]:
#Taking another look at the dataset, trying to find anything that needs to be cleaned.
df.head(100)

,Product Id,Product Name,Category,Discounted Price,Actual Price,Discount Percentage,Rating,Rating Count,User Id,User Name,Review Id,Sub Category,Brand
0,B07JW9H4J1,"Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)",Computers&Accessories,399.00,1099.00,64%,4.2,"24,269","AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBBSNLYT3ONILA,AHCTC6ULH4XB6YHDY6PCH2R772LQ,AGYHHIERNXKA6P5T7CZLXKVPT7IQ,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q,AENGU523SXMOS7JPDTW52PNNVWGQ,AEQJHCVTNINBS4FKTBGQRQTGTE5Q,AFC3FFC5PKFF5PMA52S3VCHOZ5FQ","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jaspreet singh,Khaja moin,Anand,S.ARUMUGAM","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1KD19VHEDV0OR,R3C02RMYQMK6FC,R39GQRVBUZBWGY,R2K9EDOE15QIRJ,R3OI7YT648TL8I",Accessories&Peripherals,Wayona
1,B098NS6PVG,"Ambrane Unbreakable 60W / 3A Fast Charging 1.5m Braided Type C Cable for Smartphones, Tablets, Laptops & other Type C devices, PD Technology, 480Mbps Data Sync, Quick Charge 3.0 (RCT15A, Black)",Computers&Accessories,199.00,349.00,43%,4.0,"43,994","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBXNGXZJT525AQ,AHONIZU3ICIEHQIGQ6R2VFRSBXOQ,AFPHD2CRPDZMWMBL7WXRSVYWS5JA,AEZ346GX3HJ4O4XNRPHCNHXQURMQ,AEPSWFPNECKO34PUC7I56ITGXR6Q,AHWVEHR5DYLVFTO2KF3IZATFQSWQ,AH4QT33M55677I7ISQOAKEQWACYQ","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,N/A,BharanI,sonia,Niam","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RYGGS0M09S3KY,R17KQRUTAN5DKS,R3AAQGS6HP2QUK,R1HDNOG6TO2CCA,R3PHKXYA5AFEOU",Accessories&Peripherals,Ambrane
2,B096MSW6CT,"Sounce Fast Phone Charging Cable & Data Sync USB Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini & iOS Devices",Computers&Accessories,199.00,1899.00,90%,3.9,"7,928","AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQLWQOWZ4N3OA,AHTPQRIMGUD4BYR5YIHBH3CCGEFQ,AEUVWXYP5LT7PZLLZENEO2NODPBQ,AHC7MPW55DOO6WNCOQVA2VHOD26A,AFDI6FRPFBTNBG7BAEB7JDJSMKDQ,AFQKCEEEKXCOHTDG4WUN3XPPHJQQ,AHKUUFNMBZIDLSSPA4FEHIO2EC7Q","Kunal,Himanshu,viswanath,sai niharka,saqib malik,Aashiq,Ramu Challa,Sanjay gupta","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R25X4TBMPY91LX,R27OK7G99VK0TR,R207CYDCHJJTCJ,R3PCU8XMU173BT,R1IMONDOWRNU5V",Accessories&Peripherals,Sounce
3,B08HDJ86NZ,"boAt Deuce USB 300 2 in 1 Type-C & Micro USB Stress Resistant, Tangle-Free, Sturdy Cable with 3A Fast Charging & 480mbps Data Transmission, 10000+ Bends Lifespan and Extended 1.5m Length(Martian Red)",Computers&Accessories,329.00,699.00,53%,4.2,"94,363","AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5SGCUQBP7YSCA,AH725ST5NW2Y4JZPKUNTIJCUK2BA,AHV3TXIFCJPMS4D5JATCEUR266MQ,AGWIGDEMFIIUAOXYY2QATNBSUGHA,AFSTSLQUV4EVEXWKBOLEFHL2H5YQ,AGAKDNBHY2FKX7I4ACRGILU7QL7A,AFNWJUWJRHCC6HN52KMG5AKZY37Q","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh chouhan,Ravi Siddan,Himanshu Goel,Udaykumar","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1CLH2ULIVG5U3,R2DMKIBGFKBD6R,RC89B5IAJUTR5,R3B3DDON5FH8DS,R13WAEJDI5RS36",Accessories&Peripherals,boAt
4,B08CF3B7N1,"Portronics Konnect L 1.2M Fast Charging 3A 8 Pin USB Cable with Charge & Sync Function for iPhone, iPad (Grey)",Computers&Accessories,154.00,399.00,61%,4.2,"16,905","AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZHM4QB2KPKFUQ,AFK4NJOLFSJGWLOJIUIAROJF6YVA,AFUOTYRFUXVPEBGIXVZZ7DR3CZUA,AFDLRSXKDZ6U3U3KD46SQLFGZQRA,AH5VLM66SIK7J3IRG4NY7XVOQ55A,AE3MQNNHHLUHXURL5S7IAR7JTGNQ,AFSEOFZY67MYC7UAJU264Z5NFTLA","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK,Bhargav,Durai Vignesh,N/A","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R3KK4GT934ST3I,RCFHMWUSBIJO,RDO7DACXMAJ84,R3A6MEZL3LY66Z,R1ESIEKPGAYA29",Accessories&Peripherals,Portronics
5,B08Y1TFSP6,"pTron Solero TB301 3A Type-C Data and Fast Charging Cable, Made in India, 480Mbps Data Sync, Strong and Durable 1.5-Meter Nylon Braided USB Cable for Type-C Devices for Charging Adapter (Black)",Computers&Accessories,149.00,1000.00,85%,3.9,"24,871","AEQ2YMXSZWEOHK2EHTNLOS56YTZQ,AGRVINWECNY7323CWFXZYYIZOFTQ,AHBAT6VLOXWGYDL57KHCNCLPXAKA,AF7NDY2H6JVYTSQOZP76

In [215]:
# Look at how multiple IDs are stuffed into single strings
df[['Product Id', 'User Id', 'Review Id']].head(3)

,Product Id,User Id,Review Id
0,B07JW9H4J1,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBBSNLYT3ONILA,AHCTC6ULH4XB6YHDY6PCH2R772LQ,AGYHHIERNXKA6P5T7CZLXKVPT7IQ,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q,AENGU523SXMOS7JPDTW52PNNVWGQ,AEQJHCVTNINBS4FKTBGQRQTGTE5Q,AFC3FFC5PKFF5PMA52S3VCHOZ5FQ","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1KD19VHEDV0OR,R3C02RMYQMK6FC,R39GQRVBUZBWGY,R2K9EDOE15QIRJ,R3OI7YT648TL8I"
1,B098NS6PVG,"AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBXNGXZJT525AQ,AHONIZU3ICIEHQIGQ6R2VFRSBXOQ,AFPHD2CRPDZMWMBL7WXRSVYWS5JA,AEZ346GX3HJ4O4XNRPHCNHXQURMQ,AEPSWFPNECKO34PUC7I56ITGXR6Q,AHWVEHR5DYLVFTO2KF3IZATFQSWQ,AH4QT33M55677I7ISQOAKEQWACYQ","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RYGGS0M09S3KY,R17KQRUTAN5DKS,R3AAQGS6HP2QUK,R1HDNOG6TO2CCA,R3PHKXYA5AFEOU"
2,B096MSW6CT,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQLWQOWZ4N3OA,AHTPQRIMGUD4BYR5YIHBH3CCGEFQ,AEUVWXYP5LT7PZLLZENEO2NODPBQ,AHC7MPW55DOO6WNCOQVA2VHOD26A,AFDI6FRPFBTNBG7BAEB7JDJSMKDQ,AFQKCEEEKXCOHTDG4WUN3XPPHJQQ,AHKUUFNMBZIDLSSPA4FEHIO2EC7Q","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R25X4TBMPY91LX,R27OK7G99VK0TR,R207CYDCHJJTCJ,R3PCU8XMU173BT,R1IMONDOWRNU5V"


In [218]:
#Convert comma-separated text into Python lists
for col in ['User Id','User Name','Review Id']:
    df[col] = df[col].astype(str).str.split(',')

#Filter rows where the list lengths match perfectly
valid_lengths = (df['User Id'].str.len()==df['User Name'].str.len())&\
                (df['User Id'].str.len()==df['Review Id'].str.len())
df = df[valid_lengths]
#Explode (unpivot) the lists into separate rows simultaneously
df = df.explode(['User Id', 'User Name', 'Review Id'])
#Remove duplicate Review IDs
df = df.drop_duplicates(subset=['Review Id'])

In [221]:
# Verify the unpivoted rows
df[['Product Id', 'User Id', 'Review Id']].head(50)

,Product Id,User Id,Review Id
0,B07JW9H4J1,AG3D6O4STAQKAY2UVGEUV46KN35Q,R3HXWT0LRP0NMF
0,B07JW9H4J1,AHMY5CWJMMK5BJRBBSNLYT3ONILA,R2AJM3LFTLZHFO
0,B07JW9H4J1,AHCTC6ULH4XB6YHDY6PCH2R772LQ,R6AQJGUP6P86
0,B07JW9H4J1,AGYHHIERNXKA6P5T7CZLXKVPT7IQ,R1KD19VHEDV0OR
0,B07JW9H4J1,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q,R3C02RMYQMK6FC
0,B07JW9H4J1,AENGU523SXMOS7JPDTW52PNNVWGQ,R39GQRVBUZBWGY
0,B07JW9H4J1,AEQJHCVTNINBS4FKTBGQRQTGTE5Q,R2K9EDOE15QIRJ
0,B07JW9H4J1,AFC3FFC5PKFF5PMA52S3VCHOZ5FQ,R3OI7YT648TL8I
1,B098NS6PVG,AECPFYFQVRUWC3KGNLJIOREFP5LQ,RGIQEG07R9HS2
1,B098NS6PVG,AGYYVPDD7YG7FYNBXNGXZJT525AQ,R1SMWZQ86XIN8U


In [222]:
df.to_csv('amazon-python.csv',index=False,encoding='utf-8-sig')

In [265]:
df.columns

Index(['Product Id', 'Product Name', 'Category', 'Discounted Price',
       'Actual Price', 'Discount Percentage', 'Rating', 'Rating Count',
       'User Id', 'User Name', 'Review Id', 'Sub Category', 'Brand'],
      dtype='str')

In [266]:
#Select the final columns I want for the dashboard
final_columns = [
    'Product Id', 'Product Name', 'Brand', 'Category', 'Sub Category', 
    'Discounted Price', 'Actual Price', 'Discount Percentage', 
    'Rating', 'Rating Count', 'User Id', 'User Name', 'Review Id'
]
df_final = df[final_columns].copy()
#Standardize ID column names
df_final = df_final.rename(columns={
    'Product Id': 'Product ID', 
    'User Id': 'User ID', 
    'Review Id': 'Review ID'
})

In [267]:
# Verify the final flat table is ready
print(df_final.info())

<class 'pandas.DataFrame'>
Index: 9213 entries, 0 to 1464
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Product ID           9213 non-null   str    
 1   Product Name         9213 non-null   str    
 2   Brand                9213 non-null   object 
 3   Category             9213 non-null   str    
 4   Sub Category         9213 non-null   str    
 5   Discounted Price     9213 non-null   float64
 6   Actual Price         9213 non-null   float64
 7   Discount Percentage  9213 non-null   str    
 8   Rating               9213 non-null   str    
 9   Rating Count         9211 non-null   str    
 10  User ID              9213 non-null   str    
 11  User Name            9213 non-null   str    
 12  Review ID            9213 non-null   str    
dtypes: float64(2), object(1), str(10)
memory usage: 3.1+ MB
None


In [268]:
df_final['Discount Percentage'].head(10)

0    64%
0    64%
0    64%
0    64%
0    64%
0    64%
0    64%
0    64%
1    43%
1    43%
Name: Discount Percentage, dtype: str

In [269]:
#Fix Discount Percentage (remove %, convert to decimal)
df_final['Discount Percentage'] = df_final['Discount Percentage'].str.replace('%','')
df_final['Discount Percentage'] = pd.to_numeric(df_final['Discount Percentage'])/100

In [270]:
#Verify the fix
df_final['Discount Percentage'].head(10)

0    0.64
0    0.64
0    0.64
0    0.64
0    0.64
0    0.64
0    0.64
0    0.64
1    0.43
1    0.43
Name: Discount Percentage, dtype: float64

In [271]:
#Fix Rating Count (remove commas, handle errors, fill missing with 0, convert to integer)
df_final['Rating Count'] = df_final['Rating Count'].str.replace(',', '')
df_final['Rating Count'] = pd.to_numeric(
    df_final['Rating Count'],
    errors='coerce'
).fillna(0).astype(int)

In [272]:
df_final['Rating Count'].head(10)

0    24269
0    24269
0    24269
0    24269
0    24269
0    24269
0    24269
0    24269
1    43994
1    43994
Name: Rating Count, dtype: int64

In [273]:
#Fix Rating (convert to number, forcing any weird text to NaN)
df_final['Rating'] = pd.to_numeric(df_final['Rating'],errors='coerce')

In [275]:
# Verify the final data types
print(df_final.info())

<class 'pandas.DataFrame'>
Index: 9213 entries, 0 to 1464
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Product ID           9213 non-null   str    
 1   Product Name         9213 non-null   str    
 2   Brand                9213 non-null   object 
 3   Category             9213 non-null   str    
 4   Sub Category         9213 non-null   str    
 5   Discounted Price     9213 non-null   float64
 6   Actual Price         9213 non-null   float64
 7   Discount Percentage  9213 non-null   float64
 8   Rating               9205 non-null   float64
 9   Rating Count         9213 non-null   int64  
 10  User ID              9213 non-null   str    
 11  User Name            9213 non-null   str    
 12  Review ID            9213 non-null   str    
dtypes: float64(4), int64(1), object(1), str(7)
memory usage: 3.0+ MB
None


In [276]:
#Spot on! `df_final.info()` 
#shows that the Rating column has 9,205 non-null values out of 9,213,
#meaning 8 rows are missing a rating.

In [278]:
# Check total missing values per column
df_final.isna().sum()

Product ID             0
Product Name           0
Brand                  0
Category               0
Sub Category           0
Discounted Price       0
Actual Price           0
Discount Percentage    0
Rating                 8
Rating Count           0
User ID                0
User Name              0
Review ID              0
dtype: int64

In [282]:
# Inspect the exact rows missing a Rating
df_final[df_final['Rating'].isna()]

,Product ID,Product Name,Brand,Category,Sub Category,Discounted Price,Actual Price,Discount Percentage,Rating,Rating Count,User ID,User Name,Review ID
1279,B08L12N5H1,"Eureka Forbes car Vac 100 Watts Powerful Suction Vacuum Cleaner with Washable HEPA Filter, 3 Accessories,Compact,Light Weight & Easy to use (Black and Red)",Eureka,Home&Kitchen,Kitchen&HomeAppliances,2099.0,2499.0,0.16,NaN,992,AGTDSNT2FKVYEPDPXAA673AIS44A,Divya,R2KKTKM4M9RDVJ
1279,B08L12N5H1,"Eureka Forbes car Vac 100 Watts Powerful Suction Vacuum Cleaner with Washable HEPA Filter, 3 Accessories,Compact,Light Weight & Easy to use (Black and Red)",Eureka,Home&Kitchen,Kitchen&HomeAppliances,2099.0,2499.0,0.16,NaN,992,AER2XFSWNN4LAUCJ55IY5SOMF7WA,Dr Nefario,R1O692MZOBTE79
1279,B08L12N5H1,"Eureka Forbes car Vac 100 Watts Powerful Suction Vacuum Cleaner with Washable HEPA Filter, 3 Accessories,Compact,Light Weight & Easy to use (Black and Red)",Eureka,Home&Kitchen,Kitchen&HomeAppliances,2099.0,2499.0,0.16,NaN,992,AE3MSW6H3AL6F3ZGR5LCN5AHJO6A,Deekshith,R2WRSEWL56SOS4
1279,B08L12N5H1,"Eureka Forbes car Vac 100 Watts Powerful Suction Vacuum Cleaner with Washable HEPA Filter, 3 Accessories,Compact,Light Weight & Easy to use (Black and Red)",Eureka,Home&Kitchen,Kitchen&HomeAppliances,2099.0,2499.0,0.16,NaN,992,AG5OL5WIIPJBY25HISJLM5K2UBTQ,Preeti,R3VZRQJOKCBSH4
1279,B08L12N5H1,"Eureka Forbes car Vac 100 Watts Powerful Suction Vacuum Cleaner with Washable HEPA Filter, 3 Accessories,Compact,Light Weight & Easy to use (Black and Red)",Eureka,Home&Kitchen,Kitchen&HomeAppliances,2099.0,2499.0,0.16,NaN,992,AGHFSIBYVYXUGSNYUDAHBGOIZ3KQ,Prasanth R,R2QI4626ASSCIT
1279,B08L12N5H1,"Eureka Forbes car Vac 100 Watts Powerful Suction Vacuum Cleaner with Washable HEPA Filter, 3 Accessories,Compact,Light Weight & Easy to use (Black and Red)",Eureka,Home&Kitchen,Kitchen&HomeAppliances,2099.0,2499.0,0.16,NaN,992,AHYH6AZT3U3U44CDW5Y563UYIIUA,Pradeep kashiram Tetgure.,R1TFFJ5ON6ATEO
1279,B08L12N5H1,"Eureka Forbes car Vac 100 Watts Powerful Suction Vacuum Cleaner with Washable HEPA Filter, 3 Accessories,Compact,Light Weight & Easy to use (Black and Red)",Eureka,Home&Kitchen,Kitchen&HomeAppliances,2099.0,2499.0,0.16,NaN,992,AFLOAOURRZZZGFBF7F6IKGXRB6NQ,Abhijin Janardhan,R14JK9VQCXXEKU
1279,B08L12N5H1,"Eureka Forbes car Vac 100 Watts Powerful Suction Vacuum Cleaner with Washable HEPA Filter, 3 Accessories,Compact,Light Weight & Easy to use (Black and Red)",Eureka,Home&Kitchen,Kitchen&HomeAppliances,2099.0,2499.0,0.16,NaN,992,AGNWBYEVAIII4MPQNKN3LFVOHYZQ,Prashant,R1V4J4B7RXHG8T


In [283]:
#Since all 8 missing rows belong to the same product (Eureka Forbes Car Vac), 
#I can either fill the missing rating with 
#the overall dataset median rating (so I don’t skew the stats) 
#or drop those rows. 
#Filling with the median is best to preserve the customer count.
#This is highly dependent on 
#the dataset, the management goals, the analysis I want to do, and the data source.

In [286]:
# Calculate median rating across the dataset
median_rating = df_final['Rating'].median()
print(median_rating)

4.1


In [287]:
#Fill missing ratings with the median
df_final['Rating'] = df_final['Rating'].fillna(median_rating)

In [290]:
# Verify no null values remain
df_final['Rating'].isna().sum()

np.int64(0)

In [293]:
print(df_final.info())

<class 'pandas.DataFrame'>
Index: 9213 entries, 0 to 1464
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Product ID           9213 non-null   str    
 1   Product Name         9213 non-null   str    
 2   Brand                9213 non-null   object 
 3   Category             9213 non-null   str    
 4   Sub Category         9213 non-null   str    
 5   Discounted Price     9213 non-null   float64
 6   Actual Price         9213 non-null   float64
 7   Discount Percentage  9213 non-null   float64
 8   Rating               9213 non-null   float64
 9   Rating Count         9213 non-null   int64  
 10  User ID              9213 non-null   str    
 11  User Name            9213 non-null   str    
 12  Review ID            9213 non-null   str    
dtypes: float64(4), int64(1), object(1), str(7)
memory usage: 3.0+ MB
None


In [295]:
# Convert Brand to string type
df_final['Brand'] = df_final['Brand'].astype(str)

In [298]:
df_final.dtypes

Product ID                 str
Product Name               str
Brand                      str
Category                   str
Sub Category               str
Discounted Price       float64
Actual Price           float64
Discount Percentage    float64
Rating                 float64
Rating Count             int64
User ID                    str
User Name                  str
Review ID                  str
dtype: object

In [299]:
# Check numerical bounds (min, max, mean)
print(df_final[['Discounted Price', 'Actual Price', 'Discount Percentage', 'Rating', 'Rating Count']].describe())

       Discounted Price   Actual Price  Discount Percentage       Rating  \
count       9213.000000    9213.000000          9213.000000  9213.000000   
mean        2513.021596    4550.870474             0.462788     4.085325   
std         5549.713960    9333.083076             0.214442     0.298883   
min           39.000000      39.000000             0.000000     2.000000   
25%          348.000000     845.000000             0.310000     3.900000   
50%          809.000000    1690.000000             0.480000     4.100000   
75%         1995.000000    3990.000000             0.620000     4.300000   
max        77990.000000  139900.000000             0.940000     5.000000   

        Rating Count  
count    9213.000000  
mean    14017.438619  
std     32914.802617  
min         0.000000  
25%       992.000000  
50%      3842.000000  
75%     13120.000000  
max    426973.000000  


In [300]:
df_final.to_csv('amazon-python.csv',index=False,encoding='utf-8-sig')

In [301]:
#Now let's build the dashboard. I use Streamlit.

In [308]:
#When I was building the dashboard, I noticed some mistakes in the Brand column.

In [344]:
df_final['Brand'].unique()

<ArrowStringArray>
[        'Wayona',        'Ambrane',         'Sounce',           'boAt',
     'Portronics',             'MI',        'TP-Link',   'AmazonBasics',
             'LG',       'Duracell',
 ...
         'Dynore',       'LACOPINE',        'Karcher',        'Larrito',
         'Hilton',          'Syska', 'Kitchengenix's',         'KNOWZA',
            'NGI',           'Noir']
Length: 406, dtype: str

In [321]:
#I need to show all the rows.
pd.set_option('display.max_rows',None)

In [323]:
df_final['Brand'].unique()

<ArrowStringArray>
[        'Wayona',        'Ambrane',         'Sounce',           'boAt',
     'Portronics',             'MI',        'TP-Link',   'AmazonBasics',
             'LG',       'Duracell',
 ...
         'Dynore',       'LACOPINE',        'Karcher',        'Larrito',
         'Hilton',          'Syska', 'Kitchengenix's',         'KNOWZA',
            'NGI',           'Noir']
Length: 406, dtype: str

In [327]:
#It still doesn’t work. Let’s try another solution.
for brand in df_final['Brand'].unique():
    print(brand)
#Some values need to be corrected.

Wayona
Ambrane
Sounce
boAt
Portronics
MI
TP-Link
AmazonBasics
LG
Duracell
tizum
Samsung
Flix
Acer
Tizum
OnePlus
Zoul
pTron
Amazonbasics
Mi
Wecool
D-Link
7SEVEN
VW
Tata
Airtel
Lapster
Redmi
Model-P4
Amazon
oraimo
CEDO
Pinnaclz
TCL
SWAPKART
Firestick
SKYWALL
Gizga
ZEBRONICS
LOHAYA
Gilary
Dealfreez
Isoelite
VU
Croma
Cotbolt
Electvision
King
Belkin
Remote
Hisense
iFFALCON
Saifsmart
Unknown
LRIPL
Kodak
BlueRigger
GENERIC
EGate
Realme
Syncwire
Skadioo
Sony
Storite
Karbonn
Zebronics
Time
Caldipree
Universal
Amkette
POPIO
MYVN
WZATCO
Crypo
Posh
Astigo
Caprigo
TATA
Soni
Rts
Agaro
Sansui
Hi-Mobiler
Smashtronics
SVM
CableCreation
Cubetek
KRISONS
Toshiba
Lenovo
Tuarso
PROLEGEND
WANBO
Lava
Technotech
NK
LS
ZORBES
Synqe
Bestor
Irusu
Shopoflux
EYNK
LUNAGARIYA
PRUSHTI
Aine
REDTECH
ESR
Fire-Boltt
SanDisk
Noise
Nokia
JBL
PTron
ELV
iQOO
WeCool
OPPO
Tygot
STRIFF
Oraimo
Goldmedal
HP
Spigen
Motorola
KINGONE
Tecno
Tukzer
Elv
Myvn
Newly
Kyosei
OpenTech
EN
URBN
Apple
LIRAMARK
SHREENOVA
POCO
WeRun
Tokdis
LAPSTE

In [345]:
#Strip the Brand column to remove the spaces at the beginning and the end.
df_final['Brands'] = df_final['Brand'].str.strip()

In [346]:
#Make a lowercase version of the brands
lower_map = df_final['Brand'].str.lower()
canonical = (
    df_final.groupby(lower_map)['Brand']#Group using the lowercase names
    .agg(lambda x: x.value_counts().idxmax())  # most frequent casing wins
)
df_final['Brand_clean'] = lower_map.map(canonical)#map() means "take each value 
                                            #and convert it using a rule."

In [352]:
# Identify potentially duplicate or misspelled brand names by comparing each unique brand
# with all other brands using fuzzy string matching,
#then display matches with a similarity score of 85% or higher.
uniques = df_final['Brand_clean'].unique()
suspects = []
for current_brand in uniques:
    match, score, _ = process.extractOne(
        current_brand,
        [other_brand for other_brand in uniques if other_brand != current_brand],
        scorer=fuzz.ratio
    )

    if score >= 85:
        suspects.append((current_brand, match, score))    
pd.DataFrame(suspects, columns=['brand', 'closest_match', 'score']).sort_values('score', ascending=False)

,brand,closest_match,score


In [357]:
# Flag placeholder/generic values in Brand_clean and replace them with "Unlisted/Junk".
placeholder_list = ['Unknown', 'GENERIC', 'Generic', 'Remote', 'PC', 'Kitchen', 'Room',
                     'White', 'Black', 'Green', 'Silicone', 'Time', 'Universal', 'Portable',
                     'Personal', 'Instant', 'Milk', 'House', 'Heart', 'Pick', 'Swiss',
                     'Sure', 'C', 'Sui', 'TE', 'MR.']

df_final['Brand_flagged'] = df_final['Brand_clean'].isin(placeholder_list)
df_final.loc[df_final['Brand_flagged'], 'Brand_clean'] = 'Unlisted/Junk'

In [358]:
# Manually correct specific brand names with known spelling errors.
manual_fixes={
    'Soni': 'Sony',
    "Kitchengenix's": 'Kitchengenix',
}
df_final['Brand_clean'] = df_final['Brand_clean'].replace(manual_fixes)

In [406]:
# Standardize "King" brand name and verify remaining King entries
df_final['Brand_clean'] = df_final['Brand_clean'].replace('King', 'King Shine')
df_final.loc[df_final['Brand_clean'].str.lower()=='king',['Product Name','Brand_clean']]

,Product Name,Brand_clean


In [407]:
df_final.to_csv('amazon-python.csv',index=False,encoding='utf-8-sig')

In [3]:
df = pd.read_csv('amazon-python.csv')

In [24]:
#Check the brand_clean column looking for any mistakes.
for brand in df['Brand_clean'].unique():
    print(brand)

Wayona
Ambrane
Sounce
boAt
Portronics
MI
TP-Link
AmazonBasics
LG
Duracell
Tizum
Samsung
Flix
Acer
OnePlus
Zoul
PTron
WeCool
D-Link
7SEVEN
VW
Tata
Airtel
Lapster
Redmi
Model-P4
Amazon
oraimo
CEDO
Pinnaclz
TCL
SWAPKART
Unlisted/Junk
SKYWALL
Gizga
Zebronics
LOHAYA
Gilary
Dealfreez
Isoelite
VU
Croma
Cotbolt
Electvision
King Shine
Belkin
Hisense
iFFALCON
Saifsmart
LRIPL
Kodak
BlueRigger
EGate
Realme
Syncwire
Skadioo
Sony
Storite
Karbonn
Caldipree
Amkette
POPIO
MYVN
WZATCO
Crypo
Posh
Astigo
Caprigo
Rts
AGARO
Sansui
Hi-Mobiler
Smashtronics
SVM
CableCreation
Cubetek
KRISONS
Toshiba
Lenovo
Tuarso
PROLEGEND
WANBO
Lava
Technotech
NK
LS
ZORBES
Synqe
Bestor
Irusu
Shopoflux
EYNK
LUNAGARIYA
PRUSHTI
Aine
REDTECH
ESR
Fire-Boltt
SanDisk
Noise
Nokia
JBL
ELV
iQOO
OPPO
Tygot
STRIFF
Goldmedal
HP
Spigen
Motorola
KINGONE
Tecno
Tukzer
Newly
Kyosei
OpenTech
EN
URBN
Apple
LIRAMARK
SHREENOVA
POCO
WeRun
Tokdis
Prolet
Mobilife
DYAZO
Logitech
Storio
SKE
Boult
Dell
Boya
Classmate
Seagate
SYVO
Casio
DIGITEK
Eveready
P

In [29]:
df['Brand_clean']=df['Brand_clean'].replace('Model-P4','Unlisted/Junk')

In [38]:
#Make the column widths unlimited so I can see all the columns and check the values. 
#I think they might need cleaning.
pd.set_option('display.max_colwidth',None)
df.loc[df['Brand_clean'].str.contains('SWAPKART'),['Product Name','Brand_clean']]

,Product Name,Brand_clean
512,"SWAPKART Fast Charging Cable and Data Sync USB Cable Compatible for iPhone 6/6S/7/7+/8/8+/10/11, 12, 13 Pro max iPad Air/Mini, iPod and iOS Devices (White)",SWAPKART
513,"SWAPKART Fast Charging Cable and Data Sync USB Cable Compatible for iPhone 6/6S/7/7+/8/8+/10/11, 12, 13 Pro max iPad Air/Mini, iPod and iOS Devices (White)",SWAPKART
514,"SWAPKART Fast Charging Cable and Data Sync USB Cable Compatible for iPhone 6/6S/7/7+/8/8+/10/11, 12, 13 Pro max iPad Air/Mini, iPod and iOS Devices (White)",SWAPKART
515,"SWAPKART Fast Charging Cable and Data Sync USB Cable Compatible for iPhone 6/6S/7/7+/8/8+/10/11, 12, 13 Pro max iPad Air/Mini, iPod and iOS Devices (White)",SWAPKART
516,"SWAPKART Fast Charging Cable and Data Sync USB Cable Compatible for iPhone 6/6S/7/7+/8/8+/10/11, 12, 13 Pro max iPad Air/Mini, iPod and iOS Devices (White)",SWAPKART
517,"SWAPKART Fast Charging Cable and Data Sync USB Cable Compatible for iPhone 6/6S/7/7+/8/8+/10/11, 12, 13 Pro max iPad Air/Mini, iPod and iOS Devices (White)",SWAPKART
518,"SWAPKART Fast Charging Cable and Data Sync USB Cable Compatible for iPhone 6/6S/7/7+/8/8+/10/11, 12, 13 Pro max iPad Air/Mini, iPod and iOS Devices (White)",SWAPKART
519,"SWAPKART Fast Charging Cable and Data Sync USB Cable Compatible for iPhone 6/6S/7/7+/8/8+/10/11, 12, 13 Pro max iPad Air/Mini, iPod and iOS Devices (White)",SWAPKART
2522,"SWAPKART Flexible Mobile Tabletop Stand, Metal Built, Heavy Duty Foldable Lazy Bracket Clip Mount Multi Angle Clamp for All Smartphones (Pack of 1), Multi Color",SWAPKART
2523,"SWAPKART Flexible Mobile Tabletop Stand, Metal Built, Heavy Duty Foldable Lazy Bracket Clip Mount Multi Angle Clamp for All Smartphones (Pack of 1), Multi Color",SWAPKART


In [59]:
#Some fixes.
#df.loc[df['Brand_clean'].str.lower()=='rc','Brand_clean']= 'RC PRINT'
#df[df['Brand_clean']=='RC']
#df[df['Brand_clean']=='VR']
#df[df['Brand_clean']=='INDIAS']
#df[df['Brand_clean']=='EN']
#df.loc[df['Brand_clean']=='EN','Brand_clean'] = 'EN LIGNE'
#df = df.drop(columns=['Brand_flagged'])
#df = df.drop(columns=['Brand'])
#df = df.rename(columns={'Brand_clean': 'Brand'})
#df.columns
#df['Brand']
df.to_csv('amazon-python.csv',index=False,encoding='utf-8-sig')

In [61]:
df = pd.read_csv('amazon-python.csv')

In [69]:
#df.head(100)
#I want to check the discounted price to ensure it is calculated correctly.

#Calculate expected discount percentage
expected = ((df['Actual Price']-df['Discounted Price'])/df['Actual Price'])*100
# Count total rows with incorrect percentages
(df['Discount Percentage'].round(2)!=expected.round(2)).sum()

np.int64(8871)

In [74]:
#"The Discount Percentage column clearly needs to be rebuilt."


df['Discount Percentage'] = \
(((df['Actual Price'] - df['Discounted Price']) / df['Actual Price']) * 100).round(2)

In [76]:
#Calculate expected discount percentage
expected = ((df['Actual Price']-df['Discounted Price'])/df['Actual Price'])*100
# Count total rows with incorrect percentages
(df['Discount Percentage'].round(2)!=expected.round(2)).sum()

np.int64(0)

In [78]:
#Great, the Discount Percentage is now accurate.

In [81]:
#Take another look at the data.
df.head(50)

,Product ID,Product Name,Category,Sub Category,Discounted Price,Actual Price,Discount Percentage,Rating,Rating Count,User ID,User Name,Review ID,Brand
0,B07JW9H4J1,"Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)",Computers&Accessories,Accessories&Peripherals,399.00,1099.0,63.69,4.2,24269,AG3D6O4STAQKAY2UVGEUV46KN35Q,Manav,R3HXWT0LRP0NMF,Wayona
1,B07JW9H4J1,"Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)",Computers&Accessories,Accessories&Peripherals,399.00,1099.0,63.69,4.2,24269,AHMY5CWJMMK5BJRBBSNLYT3ONILA,Adarsh gupta,R2AJM3LFTLZHFO,Wayona
2,B07JW9H4J1,"Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)",Computers&Accessories,Accessories&Peripherals,399.00,1099.0,63.69,4.2,24269,AHCTC6ULH4XB6YHDY6PCH2R772LQ,Sundeep,R6AQJGUP6P86,Wayona
3,B07JW9H4J1,"Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)",Computers&Accessories,Accessories&Peripherals,399.00,1099.0,63.69,4.2,24269,AGYHHIERNXKA6P5T7CZLXKVPT7IQ,S.Sayeed Ahmed,R1KD19VHEDV0OR,Wayona
4,B07JW9H4J1,"Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)",Computers&Accessories,Accessories&Peripherals,399.00,1099.0,63.69,4.2,24269,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q,jaspreet singh,R3C02RMYQMK6FC,Wayona
5,B07JW9H4J1,"Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)",Computers&Accessories,Accessories&Peripherals,399.00,1099.0,63.69,4.2,24269,AENGU523SXMOS7JPDTW52PNNVWGQ,Khaja moin,R39GQRVBUZBWGY,Wayona
6,B07JW9H4J1,"Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)",Computers&Accessories,Accessories&Peripherals,399.00,1099.0,63.69,4.2,24269,AEQJHCVTNINBS4FKTBGQRQTGTE5Q,Anand,R2K9EDOE15QIRJ,Wayona
7,B07JW9H4J1,"Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)",Computers&Accessories,Accessories&Peripherals,399.00,1099.0,63.69,4.2,24269,AFC3FFC5PKFF5PMA52S3VCHOZ5FQ,S.ARUMUGAM,R3OI7YT648TL8I,Wayona
8,B098NS6PVG,"Ambrane Unbreakable 60W / 3A Fast Charging 1.5m Braided Type C Cable for Smartphones, Tablets, Laptops & other Type C devices, PD Technology, 480Mbps Data Sync, Quick Charge 3.0 (RCT15A, Black)",Computers&Accessories,Accessories&Peripherals,199.00,349.0,42.98,4.0,43994,AECPFYFQVRUWC3KGNLJIOREFP5LQ,ArdKn,RGIQEG07R9HS2,Ambrane
9,B098NS6PVG,"Ambrane Unbreakable 60W / 3A Fast Charging 1.5m Braided Type C Cable for Smartphones, Tablets, Laptops & other Type C devices, PD Technology, 480Mbps Data Sync, Quick Charge 3.0 (RCT15A, Black)",Computers&Accessories,Accessories&Peripherals,199.00,349.0,42.98,4.0,43994,AGYYVPDD7YG7FYNBXNGXZJT525AQ,Nirbhay kumar,R1SMWZQ86XIN8U,Ambrane


In [83]:
df.to_csv('amazon-python.csv',index=False,encoding='utf-8-sig')

In [84]:
#I think the data is clean—not perfect, 
#but it will be great for the project's purpose
#and the analysis that will be done on it.